In [ ]:
import os, random, gc, shutil
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/nvidia-cuda-toolkit'
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121, InceptionV3, MobileNetV2, ResNet50
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, log_loss
from scipy.ndimage import binary_fill_holes, binary_opening, binary_closing, gaussian_filter

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_DIR = Path(".").resolve()
DATASET_DIR = PROJECT_DIR / "dataset"
SPLIT_DIR = PROJECT_DIR / "data_split"
MODEL_WEIGHTS_DIR = PROJECT_DIR / "model_weights"
BENCHMARK_CSV = PROJECT_DIR / "benchmark.csv"
CLASS_NAMES = ["Normal", "Stroke"]
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS = 100

print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Dataset:", DATASET_DIR)

In [ ]:
def _collect_pngs(folder: Path):
    if not folder.exists():
        return []
    return sorted(folder.glob("*.png"))

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)
SPLIT_DIR.mkdir(parents=True)

stroke_paths = sorted(
    _collect_pngs(DATASET_DIR / 'Bleeding' / 'PNG') +
    _collect_pngs(DATASET_DIR / 'Ischemia' / 'PNG')
)
normal_paths_all = _collect_pngs(DATASET_DIR / 'Normal' / 'PNG')

if not stroke_paths:
    raise FileNotFoundError(f"No stroke PNGs in {DATASET_DIR}/Bleeding/PNG or {DATASET_DIR}/Ischemia/PNG")
if not normal_paths_all:
    normal_paths_all = _collect_pngs(DATASET_DIR / 'Normal')
    if not normal_paths_all:
        raise FileNotFoundError(f"No normal PNGs in {DATASET_DIR}/Normal")

rng_prep = np.random.default_rng(SEED)
n_stroke = len(stroke_paths)
normal_pool = np.array(normal_paths_all, dtype=object)
size_to_sample = min(n_stroke, len(normal_paths_all))
normal_paths = rng_prep.choice(normal_pool, size=size_to_sample, replace=False).tolist()

print(f"Stroke pool: {len(stroke_paths)}")
print(f"Normal pool (after undersample): {len(normal_paths)}")

def _split(paths, train_r=0.70, val_r=0.15):
    arr = list(paths)
    np.random.default_rng(SEED).shuffle(arr)
    n = len(arr)
    n_train = int(n * train_r)
    n_val = int(n * val_r)
    return arr[:n_train], arr[n_train:n_train + n_val], arr[n_train + n_val:]

stroke_train, stroke_val, stroke_test = _split(stroke_paths)
normal_train, normal_val, normal_test = _split(normal_paths)

for split_name, cls in [('train', {'Stroke': stroke_train, 'Normal': normal_train}),
                          ('val', {'Stroke': stroke_val, 'Normal': normal_val}),
                          ('test', {'Stroke': stroke_test, 'Normal': normal_test})]:
    for class_name, paths in cls.items():
        dest = SPLIT_DIR / split_name / class_name
        dest.mkdir(parents=True)
        for src in paths:
            shutil.copy2(src, dest / Path(src).name)
        print(f"{split_name:5s}/{class_name:6s}: {len(paths):4d}")

In [ ]:
def make_data_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.03),
        layers.RandomZoom(0.08),
        layers.RandomContrast(0.08),
    ], name="augmentation")

def load_split_dataset(split_name: str, shuffle: bool):
    ds = tf.keras.utils.image_dataset_from_directory(
        SPLIT_DIR / split_name,
        labels="inferred", label_mode="binary",
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
        color_mode="rgb", seed=SEED, shuffle=shuffle,
    )
    if split_name != "train":
        ds = ds.cache()
    return ds.prefetch(AUTOTUNE)

train_ds = load_split_dataset("train", shuffle=True)
val_ds = load_split_dataset("val", shuffle=False)
test_ds = load_split_dataset("test", shuffle=False)

In [ ]:
TRANSFER_SPECS = {
    "ResNet50": (ResNet50, resnet_preprocess),
    "DenseNet121": (DenseNet121, densenet_preprocess),
    "MobileNetV2": (MobileNetV2, mobilenet_preprocess),
    "InceptionV3": (InceptionV3, inception_preprocess),
}

def build_transfer_model(model_name: str):
    base_constructor, preprocess_fn = TRANSFER_SPECS[model_name]
    base_model = base_constructor(
        include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,)
    )
    base_model.trainable = False
    inputs = keras.Input(shape=IMAGE_SIZE + (3,))
    x = make_data_augmentation()(inputs)
    x = layers.Lambda(preprocess_fn)(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inputs, outputs, name=model_name)

def build_alexnet():
    return keras.Sequential([
        layers.Input(shape=IMAGE_SIZE + (3,)),
        layers.Rescaling(1.0 / 255.0),
        make_data_augmentation(),
        layers.Conv2D(96, 11, strides=4, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(3, strides=2),
        layers.Conv2D(256, 5, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(3, strides=2),
        layers.Conv2D(384, 3, padding="same", activation="relu"),
        layers.Conv2D(384, 3, padding="same", activation="relu"),
        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(3, strides=2),
        layers.Flatten(),
        layers.Dense(2048, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid", dtype="float32"),
    ], name="AlexNet")

def build_model(model_name: str):
    if model_name == "AlexNet":
        return build_alexnet()
    if model_name == "HybridFusion_R50_AlexNet":
        return build_hybrid_fusion_r50_alexnet()
    if model_name == "HybridFusion_R50_AlexNet_NoPreproc":
        return build_hybrid_no_preproc()
    if model_name == "HybridFusion_R50_AlexNet_Concat":
        return build_hybrid_concat()
    if model_name == "HybridFusion_R50_AlexNet_Avg":
        return build_hybrid_average()
    return build_transfer_model(model_name)

def compile_model(model: keras.Model, learning_rate: float = 1e-4):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

BENCHMARK_MODEL_NAMES = ["AlexNet", "ResNet50", "DenseNet121", "MobileNetV2", "InceptionV3"]

In [ ]:
def collect_labels(dataset):
    labels = []
    for _, y in dataset.as_numpy_iterator():
        labels.append(np.asarray(y).reshape(-1))
    return np.concatenate(labels).astype(np.int32)

def predict_probabilities(model, dataset):
    return model.predict(dataset, verbose=0).reshape(-1).astype(np.float32)

def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(np.int32)
    y_prob = np.clip(np.asarray(y_prob).astype(np.float32), 1e-7, 1 - 1e-7)
    y_pred = (y_prob >= threshold).astype(np.int32)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
        "Sensitivity": sensitivity,
        "AUC": roc_auc_score(y_true, y_prob),
        "Loss": log_loss(y_true, y_prob, labels=[0, 1]),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

def evaluate_model_from_saved_weights(model_name: str):
    weights_path = MODEL_WEIGHTS_DIR / f"{model_name}.weights.h5"
    if not weights_path.exists():
        raise FileNotFoundError(str(weights_path))
    model = build_model(model_name)
    compile_model(model, learning_rate=1e-4)
    model.load_weights(str(weights_path))
    y_true = collect_labels(test_ds)
    y_prob = predict_probabilities(model, test_ds)
    metrics = compute_binary_metrics(y_true, y_prob, threshold=0.5)
    row = {"Model": model_name, **metrics}
    keras.backend.clear_session()
    gc.collect()
    return row

def run_full_benchmark_from_weights():
    names = BENCHMARK_MODEL_NAMES + ["HybridFusion_R50_AlexNet", "HybridFusion_R50_AlexNet_NoPreproc", "HybridFusion_R50_AlexNet_Concat", "HybridFusion_R50_AlexNet_Avg"]
    rows, missing, failed = [], [], []
    for model_name in names:
        print(f"Evaluating {model_name}...")
        try:
            rows.append(evaluate_model_from_saved_weights(model_name))
        except FileNotFoundError as err:
            missing.append((model_name, str(err)))
        except Exception as err:
            failed.append((model_name, str(err)))
    if not rows:
        raise RuntimeError("No models evaluated. Train models first.")
    benchmark_df = pd.DataFrame(rows).sort_values(["AUC", "Accuracy", "F1"], ascending=False).reset_index(drop=True)
    benchmark_df.to_csv(BENCHMARK_CSV, index=False)
    print("Saved:", BENCHMARK_CSV)
    if missing:
        for m, p in missing:
            print(f"Missing weights: {m} -> {p}")
    if failed:
        for m, msg in failed:
            print(f"Failed: {m} -> {msg}")
    return benchmark_df

In [ ]:
def get_layer_recursive(model, layer_name):
    for layer in model._flatten_layers(include_self=False, recursive=True):
        if layer.name == layer_name:
            return layer
    raise ValueError(f"Layer {layer_name} not found")

def list_conv_layer_names(model):
    return [l.name for l in model._flatten_layers(include_self=False, recursive=True) if isinstance(l, layers.Conv2D)]

def choose_gradcam_layer(model, model_name):
    conv_names = list_conv_layer_names(model)
    if model_name == "AlexNet" and len(conv_names) >= 2:
        return conv_names[-2]
    return conv_names[-1] if conv_names else None

def load_image_for_model(image_path):
    image = keras.utils.load_img(image_path, target_size=IMAGE_SIZE)
    image_array = keras.utils.img_to_array(image)
    image_tensor = np.expand_dims(image_array, axis=0)
    return image_array.astype("uint8"), image_tensor

def make_gradcampp_heatmap(image_tensor, model, layer_name):
    target_layer = get_layer_recursive(model, layer_name)
    grad_model = keras.Model(inputs=model.inputs, outputs=[target_layer.output, model.outputs[0]])
    with tf.GradientTape() as tape2:
        with tf.GradientTape() as tape1:
            with tf.GradientTape() as tape0:
                conv_out, preds = grad_model(image_tensor, training=False)
                tape0.watch(conv_out)
                tape1.watch(conv_out)
                tape2.watch(conv_out)
                score = preds[:, 0]
            first = tape0.gradient(score, conv_out)
        second = tape1.gradient(first, conv_out)
    third = tape2.gradient(second, conv_out)
    global_sum = tf.reduce_sum(conv_out, axis=(1, 2), keepdims=True)
    alpha_num = second[0]
    alpha_den = 2.0 * second[0] + third[0] * global_sum[0] + 1e-7
    alpha = alpha_num / alpha_den
    alpha = tf.nn.relu(alpha)
    weights = tf.reduce_sum(alpha * tf.nn.relu(first[0]), axis=(0, 1))
    heatmap = tf.reduce_sum(conv_out[0] * weights, axis=-1)
    heatmap = tf.nn.relu(heatmap)
    hmax = tf.reduce_max(heatmap)
    heatmap = tf.where(hmax > 0, heatmap / hmax, heatmap)
    return heatmap.numpy()

def overlay_heatmap(image_array, heatmap, alpha=0.62):
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (image_array.shape[0], image_array.shape[1])).numpy().squeeze()
    heatmap_resized = np.clip(heatmap_resized, 0.0, 1.0)
    colored_heatmap = cm.jet(heatmap_resized)[..., :3].astype("float32")
    image_float = image_array.astype("float32") / 255.0
    blended = np.clip((1 - alpha) * image_float + alpha * colored_heatmap, 0.0, 1.0)
    return (255.0 * blended).astype("uint8"), heatmap_resized

def connected_components(mask):
    h, w = mask.shape
    visited = np.zeros_like(mask, dtype=bool)
    components = []
    for y in range(h):
        for x in range(w):
            if not mask[y, x] or visited[y, x]:
                continue
            stack = [(y, x)]
            visited[y, x] = True
            pixels = []
            while stack:
                cy, cx = stack.pop()
                pixels.append((cy, cx))
                for ny in range(max(0, cy - 1), min(h, cy + 2)):
                    for nx in range(max(0, cx - 1), min(w, cx + 2)):
                        if not visited[ny, nx] and mask[ny, nx]:
                            visited[ny, nx] = True
                            stack.append((ny, nx))
            components.append(pixels)
    return components

def largest_component(mask):
    comps = connected_components(mask)
    if not comps:
        return mask
    largest = max(comps, key=len)
    out = np.zeros_like(mask, dtype=bool)
    ys, xs = zip(*largest)
    out[ys, xs] = True
    return out

def make_intracranial_mask(image_array, percentile_cutoff=30, opening_struct_size=5, closing_struct_size=7):
    gray = image_array.astype(np.float32).mean(axis=-1)
    nonzero = gray[gray > 0]
    if nonzero.size < 20:
        return np.ones_like(gray, dtype=bool)
    cutoff = np.percentile(nonzero, percentile_cutoff)
    raw = gray > cutoff
    raw = binary_opening(raw, structure=np.ones((opening_struct_size, opening_struct_size), dtype=bool))
    raw = largest_component(raw)
    raw = binary_fill_holes(raw)
    raw = binary_closing(raw, structure=np.ones((closing_struct_size, closing_struct_size), dtype=bool))
    return raw.astype(bool)

def outside_heat_ratio(heatmap_resized, intracranial_mask):
    heat = np.asarray(heatmap_resized, dtype=np.float32)
    total = float(np.sum(heat)) + 1e-8
    outside = float(np.sum(heat * (~intracranial_mask).astype(np.float32)))
    return outside / total

def extract_roi_bboxes(heatmap_resized, threshold=0.30, percentile=84, max_regions=3):
    max_value = float(np.max(heatmap_resized))
    if max_value <= 1e-8:
        return []
    positive = heatmap_resized[heatmap_resized > 0]
    if positive.size < 20:
        return []
    cutoff = max(threshold * max_value, np.percentile(positive, percentile))
    comps = connected_components(heatmap_resized >= cutoff)
    candidates = []
    for pixels in comps:
        ys = np.array([p[0] for p in pixels])
        xs = np.array([p[1] for p in pixels])
        if len(xs) < 4:
            continue
        score = float(np.sum(heatmap_resized[ys, xs]))
        bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
        candidates.append((score, bbox))
    candidates.sort(key=lambda x: x[0], reverse=True)
    return [bbox for _, bbox in candidates[:max_regions]]

In [ ]:
def build_hybrid_fusion_r50_alexnet():
    resnet_backbone = ResNet50(include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,))
    resnet_backbone.trainable = False
    inputs = keras.Input(shape=IMAGE_SIZE + (3,))
    augmented = make_data_augmentation()(inputs)
    a = layers.Lambda(resnet_preprocess)(augmented)
    a = resnet_backbone(a, training=False)
    a = layers.GlobalAveragePooling2D()(a)
    b = layers.Rescaling(1.0 / 255.0)(augmented)
    b = layers.Conv2D(96, 11, strides=4, padding="same", activation="relu")(b)
    b = layers.BatchNormalization()(b)
    b = layers.MaxPooling2D(3, strides=2)(b)
    b = layers.Conv2D(256, 5, padding="same", activation="relu")(b)
    b = layers.BatchNormalization()(b)
    b = layers.MaxPooling2D(3, strides=2)(b)
    b = layers.Conv2D(256, 3, padding="same", activation="relu")(b)
    b = layers.GlobalAveragePooling2D()(b)
    merged = layers.Concatenate()([a, b])
    gate = layers.Dense(merged.shape[-1], activation="sigmoid")(merged)
    fused = layers.Multiply()([merged, gate])
    x = layers.Dense(512, activation="relu")(fused)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inputs, outputs, name="HybridFusion_R50_AlexNet")

HYBRID_NAME = "HybridFusion_R50_AlexNet"

In [ ]:
def build_hybrid_no_preproc():
    resnet_backbone = ResNet50(include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,))
    resnet_backbone.trainable = False
    inputs = keras.Input(shape=IMAGE_SIZE + (3,))
    a = layers.Lambda(resnet_preprocess)(inputs)
    a = resnet_backbone(a, training=False)
    a = layers.GlobalAveragePooling2D()(a)
    b = layers.Rescaling(1.0 / 255.0)(inputs)
    b = layers.Conv2D(96, 11, strides=4, padding="same", activation="relu")(b)
    b = layers.BatchNormalization()(b)
    b = layers.MaxPooling2D(3, strides=2)(b)
    b = layers.Conv2D(256, 5, padding="same", activation="relu")(b)
    b = layers.BatchNormalization()(b)
    b = layers.MaxPooling2D(3, strides=2)(b)
    b = layers.Conv2D(256, 3, padding="same", activation="relu")(b)
    b = layers.GlobalAveragePooling2D()(b)
    merged = layers.Concatenate()([a, b])
    gate = layers.Dense(merged.shape[-1], activation="sigmoid")(merged)
    fused = layers.Multiply()([merged, gate])
    x = layers.Dense(512, activation="relu")(fused)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inputs, outputs, name="HybridFusion_R50_AlexNet_NoPreproc")

def build_hybrid_concat():
    resnet_backbone = ResNet50(include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,))
    resnet_backbone.trainable = False
    inputs = keras.Input(shape=IMAGE_SIZE + (3,))
    augmented = make_data_augmentation()(inputs)
    a = layers.Lambda(resnet_preprocess)(augmented)
    a = resnet_backbone(a, training=False)
    a = layers.GlobalAveragePooling2D()(a)
    b = layers.Rescaling(1.0 / 255.0)(augmented)
    b = layers.Conv2D(96, 11, strides=4, padding="same", activation="relu")(b)
    b = layers.BatchNormalization()(b)
    b = layers.MaxPooling2D(3, strides=2)(b)
    b = layers.Conv2D(256, 5, padding="same", activation="relu")(b)
    b = layers.BatchNormalization()(b)
    b = layers.MaxPooling2D(3, strides=2)(b)
    b = layers.Conv2D(256, 3, padding="same", activation="relu")(b)
    b = layers.GlobalAveragePooling2D()(b)
    merged = layers.Concatenate()([a, b])
    x = layers.Dense(512, activation="relu")(merged)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inputs, outputs, name="HybridFusion_R50_AlexNet_Concat")

def build_hybrid_average():
    resnet_backbone = ResNet50(include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,))
    resnet_backbone.trainable = False
    inputs = keras.Input(shape=IMAGE_SIZE + (3,))
    augmented = make_data_augmentation()(inputs)
    a = layers.Lambda(resnet_preprocess)(augmented)
    a = resnet_backbone(a, training=False)
    a = layers.GlobalAveragePooling2D()(a)
    b = layers.Rescaling(1.0 / 255.0)(augmented)
    b = layers.Conv2D(96, 11, strides=4, padding="same", activation="relu")(b)
    b = layers.BatchNormalization()(b)
    b = layers.MaxPooling2D(3, strides=2)(b)
    b = layers.Conv2D(256, 5, padding="same", activation="relu")(b)
    b = layers.BatchNormalization()(b)
    b = layers.MaxPooling2D(3, strides=2)(b)
    b = layers.Conv2D(256, 3, padding="same", activation="relu")(b)
    b = layers.GlobalAveragePooling2D()(b)
    a_proj = layers.Dense(512, activation="relu")(a)
    b_proj = layers.Dense(512, activation="relu")(b)
    merged = layers.Average()([a_proj, b_proj])
    x = layers.Dense(512, activation="relu")(merged)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inputs, outputs, name="HybridFusion_R50_AlexNet_Avg")

HYBRID_NO_PREPROC_NAME = "HybridFusion_R50_AlexNet_NoPreproc"
HYBRID_CONCAT_NAME = "HybridFusion_R50_AlexNet_Concat"
HYBRID_AVG_NAME = "HybridFusion_R50_AlexNet_Avg"
ABLATION_VARIANT_NAMES = [HYBRID_NO_PREPROC_NAME, HYBRID_CONCAT_NAME, HYBRID_AVG_NAME]


In [ ]:
MODEL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

for model_name in BENCHMARK_MODEL_NAMES:
    weights_path = MODEL_WEIGHTS_DIR / f'{model_name}.weights.h5'
    if weights_path.exists():
        print(f'{model_name}: weights exist, skipping.')
        continue
    print(f'\n--- Training: {model_name} ---')
    model = build_model(model_name)
    compile_model(model, learning_rate=1e-4)
    model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.3, patience=2, min_lr=1e-6),
        ],
        verbose=1,
    )
    model.save_weights(str(weights_path))
    print(f'Saved {weights_path}')
    keras.backend.clear_session()
    gc.collect()

In [ ]:
hybrid_weights = MODEL_WEIGHTS_DIR / f'{HYBRID_NAME}.weights.h5'
hybrid_model = build_hybrid_fusion_r50_alexnet()
compile_model(hybrid_model)

if not hybrid_weights.exists():
    print(f'--- Training: {HYBRID_NAME} (Phase 1) ---')
    hybrid_model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.3, patience=2, min_lr=1e-6),
        ],
        verbose=1,
    )
    hybrid_model.save_weights(str(hybrid_weights))
    print(f'Saved {hybrid_weights}')
else:
    hybrid_model.load_weights(str(hybrid_weights))
    print(f'Loaded existing {hybrid_weights}')

In [ ]:
print('--- Phase 2: Fine-tune conv5_block (LR=1e-5) ---')
resnet_ft = ResNet50(include_top=False, weights='imagenet', input_shape=IMAGE_SIZE + (3,))
for layer in resnet_ft.layers:
    layer.trainable = False
for layer in resnet_ft.layers:
    if 'conv5_block' in layer.name:
        layer.trainable = True

_inp = keras.Input(shape=IMAGE_SIZE + (3,))
_aug = make_data_augmentation()(_inp)
_a = layers.Lambda(resnet_preprocess)(_aug)
_a = resnet_ft(_a, training=False)
_a = layers.GlobalAveragePooling2D()(_a)
_b = layers.Rescaling(1.0 / 255.0)(_aug)
_b = layers.Conv2D(96, 11, strides=4, padding='same', activation='relu')(_b)
_b = layers.BatchNormalization()(_b)
_b = layers.MaxPooling2D(3, strides=2)(_b)
_b = layers.Conv2D(256, 5, padding='same', activation='relu')(_b)
_b = layers.BatchNormalization()(_b)
_b = layers.MaxPooling2D(3, strides=2)(_b)
_b = layers.Conv2D(256, 3, padding='same', activation='relu')(_b)
_b = layers.GlobalAveragePooling2D()(_b)
_merged = layers.Concatenate()([_a, _b])
_gate = layers.Dense(_merged.shape[-1], activation='sigmoid')(_merged)
_fused = layers.Multiply()([_merged, _gate])
_x = layers.Dense(512, activation='relu')(_fused)
_x = layers.Dropout(0.35)(_x)
_x = layers.Dense(128, activation='relu')(_x)
_x = layers.Dropout(0.25)(_x)
_out = layers.Dense(1, activation='sigmoid', dtype='float32')(_x)
ft_hybrid = keras.Model(_inp, _out, name=HYBRID_NAME)

ft_hybrid.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0),
    loss='binary_crossentropy',
    metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')],
)
ft_hybrid.load_weights(str(hybrid_weights))

ft_hybrid.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.3, patience=2, min_lr=1e-7),
    ],
    verbose=1,
)

y_true = collect_labels(test_ds)
m_ft = compute_binary_metrics(y_true, predict_probabilities(ft_hybrid, test_ds))
m_p1 = compute_binary_metrics(y_true, predict_probabilities(hybrid_model, test_ds))

print(f'Phase 1   -- Acc: {m_p1["Accuracy"]:.4f} | AUC: {m_p1["AUC"]:.4f} | F1: {m_p1["F1"]:.4f}')
print(f'Phase 2   -- Acc: {m_ft["Accuracy"]:.4f} | AUC: {m_ft["AUC"]:.4f} | F1: {m_ft["F1"]:.4f}')

if m_ft['AUC'] >= m_p1['AUC']:
    ft_hybrid.save_weights(str(hybrid_weights))
    hyb_model_best = ft_hybrid
    print(f'Phase 2 saved. AUC {m_p1["AUC"]:.4f} -> {m_ft["AUC"]:.4f}')
else:
    hyb_model_best = hybrid_model
    print(f'Phase 1 kept (Phase 2 AUC {m_ft["AUC"]:.4f} < {m_p1["AUC"]:.4f})')
gc.collect()

In [ ]:
print('--- Phase 3: Deep fine-tune conv4+conv5 (LR=5e-6) ---')
hyb_model_best.load_weights(str(hybrid_weights))
for layer in hyb_model_best.layers:
    if layer.name == 'resnet50':
        layer.trainable = True
        for sub_layer in layer.layers:
            if any(block in sub_layer.name for block in ['conv5_block', 'conv4_block']):
                sub_layer.trainable = True
            else:
                sub_layer.trainable = False

hyb_model_best.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6, clipnorm=1.0),
    loss='binary_crossentropy',
    metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')],
)

hyb_model_best.fit(
    train_ds, validation_data=val_ds, epochs=50,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=3, min_lr=1e-8),
    ],
    verbose=1,
)
hyb_model_best.save_weights(str(hybrid_weights))

In [ ]:
# === Train ablation variants (Phase 1 only) ===
for name in ABLATION_VARIANT_NAMES:
    weights_path = MODEL_WEIGHTS_DIR / f"{name}.weights.h5"
    if weights_path.exists():
        print(f"{name}: weights exist, skipping.")
        continue
    print(f"\n--- Training: {name} (Phase 1) ---")
    model = build_model(name)
    compile_model(model, learning_rate=1e-4)
    model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.3, patience=2, min_lr=1e-6),
        ],
        verbose=1,
    )
    model.save_weights(str(weights_path))
    print(f"Saved {weights_path}")
    keras.backend.clear_session()
    gc.collect()


In [ ]:
benchmark_df = run_full_benchmark_from_weights()
display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'Specificity', 'Sensitivity', 'AUC', 'Loss']
print(benchmark_df[display_cols].to_string(index=False))

In [ ]:
plot_df = benchmark_df.melt(id_vars='Model', value_vars=['Accuracy', 'AUC', 'F1'], var_name='Metric', value_name='Score')
plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x='Metric', y='Score', hue='Model', palette='viridis')
plt.title('Model Comparison: HybridFusion vs Baselines')
plt.ylim(0.7, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# === Ablation Study: Gated vs Variants (Original Dataset) ===
print("\n" + "="*70)
print("ABLATION STUDY: Gated Fusion vs Variants")
print("="*70)

# Hardcoded values from bm.pdf for known models
bm_data = {
    "AlexNet":                   {"AUC": 0.957573, "Accuracy": 0.877246, "F1": 0.869010, "Precision": 0.931507, "Recall": 0.814371, "Specificity": 0.940120, "Sensitivity": 0.814371},
    "ResNet50":                  {"AUC": 0.905187, "Accuracy": 0.823353, "F1": 0.805281, "Precision": 0.897059, "Recall": 0.730539, "Specificity": 0.916168, "Sensitivity": 0.730539},
    "HybridFusion_R50_AlexNet":   {"AUC": 0.960271, "Accuracy": 0.892216, "F1": 0.889231, "Precision": 0.914557, "Recall": 0.865269, "Specificity": 0.919162, "Sensitivity": 0.865269},
}

# Evaluate ablation variants
ablation_rows = []
for name in ABLATION_VARIANT_NAMES:
    weights_path = MODEL_WEIGHTS_DIR / f"{name}.weights.h5"
    if weights_path.exists():
        try:
            row = evaluate_model_from_saved_weights(name)
            ablation_rows.append(row)
            print(f"{name}: AUC={row['AUC']:.4f}  Acc={row['Accuracy']:.4f}  F1={row['F1']:.4f}")
        except Exception as e:
            print(f"{name}: Evaluation failed - {e}")
    else:
        print(f"{name}: No weights found at {weights_path}")

# Combine all results
all_rows = []
for model_name in ["AlexNet", "ResNet50", "HybridFusion_R50_AlexNet"]:
    metrics = bm_data[model_name]
    all_rows.append({"Model": model_name, **metrics})
all_rows.extend(ablation_rows)

ablation_df = pd.DataFrame(all_rows)
display_cols = ["Model", "Accuracy", "Precision", "Recall", "F1", "Specificity", "Sensitivity", "AUC"]
print("\nAblation Study: 6-way Comparison")
print("-"*70)
print(ablation_df[display_cols].to_string(index=False))

ablation_df.to_excel(PROJECT_DIR / "ablation.xlsx", index=False)
print("\nSaved ablation.xlsx")


In [ ]:
hybrid_model_final = build_hybrid_fusion_r50_alexnet()
compile_model(hybrid_model_final)
hybrid_model_final.load_weights(str(hybrid_weights))
HYBRID_LAYER = choose_gradcam_layer(hybrid_model_final, HYBRID_NAME)
if HYBRID_LAYER is None:
    HYBRID_LAYER = 'conv2d_22'

def make_brain_tissue_mask(image_array, intracranial_mask):
    return intracranial_mask

def smooth_heatmap(heatmap, sigma=1.2):
    return gaussian_filter(heatmap, sigma=sigma)

def extract_roi_bboxes_in_tissue(hotspot, tissue_mask, threshold=0.45, percentile=92, max_regions=1):
    masked = hotspot * tissue_mask.astype(np.float32)
    return extract_roi_bboxes(masked, threshold=threshold, percentile=percentile, max_regions=max_regions)

def roi_heat_agreement(rois, heat):
    if not rois:
        return 0.0
    total = float(np.sum(heat)) + 1e-8
    inside = 0.0
    for (x1, y1, x2, y2) in rois:
        inside += float(np.sum(heat[y1:y2 + 1, x1:x2 + 1]))
    return inside / total

def focused_hotspot(masked_heat, percentile=88):
    if np.any(masked_heat > 0):
        active = masked_heat[masked_heat > 0]
        cutoff = np.percentile(active, percentile)
        return np.where(masked_heat >= cutoff, masked_heat, 0.0).astype(np.float32)
    return masked_heat.astype(np.float32)

def calculate_mean_ohr(class_name, sample_sizes=[100, 200, 300]):
    test_dir = SPLIT_DIR / 'test' / class_name
    if not test_dir.exists():
        print(f"No test dir: {test_dir}")
        return {n: 0.0 for n in sample_sizes}
    valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
    paths = sorted([p for p in test_dir.iterdir() if p.suffix.lower() in valid_ext])
    if not paths:
        print(f"No images for {class_name}")
        return {n: 0.0 for n in sample_sizes}
    results = {}
    local_rng = random.Random(SEED)
    for n in sample_sizes:
        n_to_sample = min(n, len(paths))
        sampled = local_rng.sample(paths, k=n_to_sample)
        ohr_values = []
        for p in sampled:
            try:
                image_array, image_tensor = load_image_for_model(p)
                raw_heatmap = make_gradcampp_heatmap(image_tensor, hybrid_model_final, HYBRID_LAYER)
                _, heatmap_resized = overlay_heatmap(image_array, raw_heatmap)
                intracranial_mask = make_intracranial_mask(image_array)
                ohr = outside_heat_ratio(heatmap_resized, intracranial_mask)
                ohr_values.append(ohr)
            except Exception as e:
                continue
        results[n] = np.mean(ohr_values) if ohr_values else 0.0
        print(f"  {class_name:6s} @ n={n}: {results[n]:.4f}")
    return results

stroke_ohr = calculate_mean_ohr('Stroke')
normal_ohr = calculate_mean_ohr('Normal')

ohr_rows = [{'Sample Size': n, 'Mean OHR (Stroke)': f'{stroke_ohr.get(n, 0.0):.4f}',
             'Mean OHR (Normal)': f'{normal_ohr.get(n, 0.0):.4f}'} for n in [100, 200, 300]]
display(pd.DataFrame(ohr_rows))

In [ ]:
run_seed = int.from_bytes(os.urandom(8), "big")
rng = random.Random(run_seed)
valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def get_images(class_name):
    return sorted([p for p in (SPLIT_DIR / "test" / class_name).iterdir() if p.suffix.lower() in valid_ext])

stroke_paths = get_images("Stroke")
normal_paths = get_images("Normal")

normal_pool = rng.sample(normal_paths, k=min(30, len(normal_paths)))
normal_scored = []
for image_path in normal_pool:
    image_array, image_tensor = load_image_for_model(image_path)
    prob = float(hybrid_model_final.predict(image_tensor, verbose=0).ravel()[0])
    normal_scored.append({"path": image_path, "prob": prob, "image_array": image_array})
normal_scored.sort(key=lambda x: x["prob"])
normal_case = normal_scored[0]

stroke_pool = rng.sample(stroke_paths, k=min(100, len(stroke_paths)))
stroke_scored = []
for image_path in stroke_pool:
    image_array, image_tensor = load_image_for_model(image_path)
    prob = float(hybrid_model_final.predict(image_tensor, verbose=0).ravel()[0])
    if prob < 0.70:
        continue
    raw_heatmap = make_gradcampp_heatmap(image_tensor, hybrid_model_final, HYBRID_LAYER)
    _, heatmap_resized = overlay_heatmap(image_array, raw_heatmap)
    heatmap_smooth = smooth_heatmap(heatmap_resized, sigma=1.2)
    intracranial_mask = make_intracranial_mask(image_array)
    tissue_mask = make_brain_tissue_mask(image_array, intracranial_mask)
    masked_heat = heatmap_smooth * tissue_mask.astype(np.float32)
    hotspot = focused_hotspot(masked_heat, percentile=88)
    rois = extract_roi_bboxes_in_tissue(hotspot, tissue_mask)
    if rois:
        stroke_scored.append({"path": image_path, "prob": prob, "rois": rois,
                              "image_array": image_array, "hotspot": hotspot})

stroke_scored.sort(key=lambda x: x["prob"], reverse=True)
if stroke_scored:
    stroke_case = stroke_scored[0]
    fig_n, ax_n = plt.subplots(1, 1, figsize=(5, 5))
    ax_n.imshow(normal_case["image_array"], cmap="gray")
    ax_n.set_title(f"Normal | p(stroke)={normal_case['prob']:.4f}")
    ax_n.axis("off")
    plt.show()

    fig_s, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(stroke_case["image_array"], cmap="gray")
    for (x1, y1, x2, y2) in stroke_case["rois"]:
        axes[0].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=2))
    axes[0].set_title("Stroke: Original + ROI")
    axes[0].axis("off")
    axes[1].imshow(stroke_case["image_array"], cmap="gray")
    axes[1].imshow(stroke_case["hotspot"], cmap="hot", alpha=0.5)
    axes[1].set_title(f"Stroke: Grad-CAM++ | p={stroke_case['prob']:.4f}")
    axes[1].axis("off")
    plt.show()
else:
    print("No stroke cases with prob>=0.70 found.")

In [ ]:
# === ABLATION STUDY: Train on trainset/, test on original test set ===
TRAINSET_DIR = PROJECT_DIR / "trainset"
TRAINSET_SPLIT_DIR = PROJECT_DIR / "trainset_split"
MODEL_WEIGHTS_TRAINSET_DIR = PROJECT_DIR / "model_weights_trainset"

if TRAINSET_SPLIT_DIR.exists():
    shutil.rmtree(TRAINSET_SPLIT_DIR)
TRAINSET_SPLIT_DIR.mkdir(parents=True)

def _collect_jpgs(folder: Path):
    if not folder.exists():
        return []
    return sorted(folder.glob("*.jpg"))

stroke_train_paths = _collect_jpgs(TRAINSET_DIR / 'Stroke')
normal_train_paths = _collect_jpgs(TRAINSET_DIR / 'Normal')
print(f"trainset Stroke: {len(stroke_train_paths)}, Normal: {len(normal_train_paths)}")

rng_ts = np.random.default_rng(SEED)
def _split_ts(paths, train_r=0.70, val_r=0.15):
    arr = list(paths)
    rng_ts.shuffle(arr)
    n = len(arr)
    n_train = int(n * train_r)
    n_val = int(n * val_r)
    return arr[:n_train], arr[n_train:n_train + n_val], arr[n_train + n_val:]

stroke_ts_train, stroke_ts_val, stroke_ts_test = _split_ts(stroke_train_paths)
normal_ts_train, normal_ts_val, normal_ts_test = _split_ts(normal_train_paths)

for split_name, cls in [('train', {'Stroke': stroke_ts_train, 'Normal': normal_ts_train}),
                          ('val', {'Stroke': stroke_ts_val, 'Normal': normal_ts_val})]:
    for class_name, paths in cls.items():
        dest = TRAINSET_SPLIT_DIR / split_name / class_name
        dest.mkdir(parents=True)
        for src in paths:
            shutil.copy2(src, dest / Path(src).name)
        print(f"{split_name:5s}/{class_name:6s}: {len(paths):4d}")

In [ ]:
# Load trainset splits for training
trainset_train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAINSET_SPLIT_DIR / 'train',
    labels="inferred", label_mode="binary",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    color_mode="rgb", seed=SEED, shuffle=True,
).prefetch(AUTOTUNE)

trainset_val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAINSET_SPLIT_DIR / 'val',
    labels="inferred", label_mode="binary",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    color_mode="rgb", seed=SEED, shuffle=False,
).cache().prefetch(AUTOTUNE)

print(f"Trainset train batches: {len(trainset_train_ds)}, val batches: {len(trainset_val_ds)}")
print(f"Original test batches: {len(test_ds)}")

In [ ]:
# === Train all models on trainset ===
MODEL_WEIGHTS_TRAINSET_DIR.mkdir(parents=True, exist_ok=True)

for model_name in BENCHMARK_MODEL_NAMES:
    weights_path = MODEL_WEIGHTS_TRAINSET_DIR / f'{model_name}.weights.h5'
    if weights_path.exists():
        print(f'{model_name}: weights exist, skipping.')
        continue
    print(f'\n--- Training {model_name} on trainset ---')
    model = build_model(model_name)
    compile_model(model, learning_rate=1e-4)
    model.fit(
        trainset_train_ds, validation_data=trainset_val_ds, epochs=EPOCHS,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.3, patience=2, min_lr=1e-6),
        ],
        verbose=1,
    )
    model.save_weights(str(weights_path))
    print(f'Saved {weights_path}')
    keras.backend.clear_session()
    gc.collect()

In [ ]:
# === Train Hybrid on trainset (Phase 1) ===
HYBRID_WEIGHTS_TRAINSET = MODEL_WEIGHTS_TRAINSET_DIR / f'{HYBRID_NAME}.weights.h5'
hybrid_ts = build_hybrid_fusion_r50_alexnet()
compile_model(hybrid_ts)

if not HYBRID_WEIGHTS_TRAINSET.exists():
    print(f'--- Training {HYBRID_NAME} on trainset (Phase 1) ---')
    hybrid_ts.fit(
        trainset_train_ds, validation_data=trainset_val_ds, epochs=EPOCHS,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.3, patience=2, min_lr=1e-6),
        ],
        verbose=1,
    )
    hybrid_ts.save_weights(str(HYBRID_WEIGHTS_TRAINSET))
    print(f'Saved {HYBRID_WEIGHTS_TRAINSET}')
else:
    hybrid_ts.load_weights(str(HYBRID_WEIGHTS_TRAINSET))
    print(f'Loaded existing {HYBRID_WEIGHTS_TRAINSET}')

# Phase 2 & 3: Skip for ablation (trainset is smaller, fine-tuning on original data would leak)
hybrid_ts_best = hybrid_ts

In [ ]:
# === Benchmark trainset models on ORIGINAL test set ===
BENCHMARK_V2_CSV = PROJECT_DIR / "benchmark_v2.csv"

def evaluate_trainset_model(model_name: str):
    weights_path = MODEL_WEIGHTS_TRAINSET_DIR / f"{model_name}.weights.h5"
    if not weights_path.exists():
        raise FileNotFoundError(str(weights_path))
    if model_name == HYBRID_NAME:
        model = build_hybrid_fusion_r50_alexnet()
    else:
        model = build_model(model_name)
    compile_model(model, learning_rate=1e-4)
    model.load_weights(str(weights_path))
    y_true = collect_labels(test_ds)
    y_prob = predict_probabilities(model, test_ds)
    metrics = compute_binary_metrics(y_true, y_prob, threshold=0.5)
    row = {"Model": model_name, **metrics}
    keras.backend.clear_session()
    gc.collect()
    return row

def run_benchmark_v2():
    names = BENCHMARK_MODEL_NAMES + ["HybridFusion_R50_AlexNet"]
    rows, missing, failed = [], [], []
    for model_name in names:
        print(f"Evaluating {model_name} (trainset) on original test...")
        try:
            rows.append(evaluate_trainset_model(model_name))
        except FileNotFoundError as err:
            missing.append((model_name, str(err)))
        except Exception as err:
            failed.append((model_name, str(err)))
    if not rows:
        raise RuntimeError("No models evaluated. Train trainset models first.")
    benchmark_v2_df = pd.DataFrame(rows).sort_values(["AUC", "Accuracy", "F1"], ascending=False).reset_index(drop=True)
    benchmark_v2_df.to_csv(BENCHMARK_V2_CSV, index=False)
    print("Saved:", BENCHMARK_V2_CSV)
    if missing:
        for m, p in missing:
            print(f"Missing weights: {m} -> {p}")
    if failed:
        for m, msg in failed:
            print(f"Failed: {m} -> {msg}")
    return benchmark_v2_df

benchmark_v2_df = run_benchmark_v2()
display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'Specificity', 'Sensitivity', 'AUC', 'Loss']
print("\n=== BENCHMARK V2 (trained on trainset, tested on original test) ===")
print(benchmark_v2_df[display_cols].to_string(index=False))

In [ ]:
# === OHR on original test set using trainset-trained Hybrid ===
hybrid_ts_final = build_hybrid_fusion_r50_alexnet()
compile_model(hybrid_ts_final)
hybrid_ts_final.load_weights(str(HYBRID_WEIGHTS_TRAINSET))
HYBRID_TS_LAYER = choose_gradcam_layer(hybrid_ts_final, HYBRID_NAME)
if HYBRID_TS_LAYER is None:
    HYBRID_TS_LAYER = 'conv2d_22'

def calculate_mean_ohr_v2(class_name, sample_sizes=[100, 200, 300]):
    test_dir = SPLIT_DIR / 'test' / class_name
    if not test_dir.exists():
        print(f"No test dir: {test_dir}")
        return {n: 0.0 for n in sample_sizes}
    valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
    paths = sorted([p for p in test_dir.iterdir() if p.suffix.lower() in valid_ext])
    if not paths:
        print(f"No images for {class_name}")
        return {n: 0.0 for n in sample_sizes}
    results = {}
    local_rng = random.Random(SEED)
    for n in sample_sizes:
        n_to_sample = min(n, len(paths))
        sampled = local_rng.sample(paths, k=n_to_sample)
        ohr_values = []
        for p in sampled:
            try:
                image_array, image_tensor = load_image_for_model(p)
                raw_heatmap = make_gradcampp_heatmap(image_tensor, hybrid_ts_final, HYBRID_TS_LAYER)
                _, heatmap_resized = overlay_heatmap(image_array, raw_heatmap)
                intracranial_mask = make_intracranial_mask(image_array)
                ohr = outside_heat_ratio(heatmap_resized, intracranial_mask)
                ohr_values.append(ohr)
            except Exception as e:
                continue
        results[n] = np.mean(ohr_values) if ohr_values else 0.0
        print(f"  {class_name:6s} @ n={n}: {results[n]:.4f}")
    return results

import json
print("\n=== OHR V2 (trainset-trained model on original test set) ===")
stroke_ohr_v2 = calculate_mean_ohr_v2('Stroke')
normal_ohr_v2 = calculate_mean_ohr_v2('Normal')

ohr_v2_rows = [{'Sample Size': n, 'Mean OHR (Stroke)': f'{stroke_ohr_v2.get(n, 0.0):.4f}',
                'Mean OHR (Normal)': f'{normal_ohr_v2.get(n, 0.0):.4f}'} for n in [100, 200, 300]]
ohr_v2_df = pd.DataFrame(ohr_v2_rows)
print("\n=== OHR V2 RESULTS ===")
print(ohr_v2_df.to_string(index=False))

ohr_v2_df.to_excel(PROJECT_DIR / "ohr_v2.xlsx", index=False)
print(f"Saved OHR V2 results to {PROJECT_DIR / 'ohr_v2.xlsx'}")

ohr_v2_data = {"Stroke": stroke_ohr_v2, "Normal": normal_ohr_v2}
with open(PROJECT_DIR / "ohr_results_v2.json", "w") as f:
    json.dump(ohr_v2_data, f, indent=2)
print(f"Saved OHR V2 JSON to {PROJECT_DIR / 'ohr_results_v2.json'}")

In [ ]:
# === Compare Original vs Trainset ===
print("\n" + "="*70)
print("COMPARISON: Original Training vs Trainset Ablation")
print("="*70)

# Load original benchmark if exists
orig_bench = pd.read_csv(BENCHMARK_CSV) if BENCHMARK_CSV.exists() else None

if orig_bench is not None:
    merge_df = benchmark_v2_df[['Model', 'AUC', 'Accuracy', 'F1']].rename(
        columns={'AUC': 'Trainset_AUC', 'Accuracy': 'Trainset_Acc', 'F1': 'Trainset_F1'})
    orig_plot = orig_bench[['Model', 'AUC', 'Accuracy', 'F1']].rename(
        columns={'AUC': 'Original_AUC', 'Accuracy': 'Original_Acc', 'F1': 'Original_F1'})
    cmp = pd.merge(orig_plot, merge_df, on='Model')
    cmp['AUC_Delta'] = cmp['Trainset_AUC'] - cmp['Original_AUC']
    cmp['Acc_Delta'] = cmp['Trainset_Acc'] - cmp['Original_Acc']
    cmp['F1_Delta'] = cmp['Trainset_F1'] - cmp['Original_F1']
    print(cmp.to_string(index=False))
    
    # Plot comparison
    cmp_melt = cmp.melt(id_vars='Model', 
        value_vars=['Original_AUC', 'Trainset_AUC', 'Original_Acc', 'Trainset_Acc', 'Original_F1', 'Trainset_F1'],
        var_name='Metric', value_name='Score')
    plt.figure(figsize=(14, 6))
    sns.barplot(data=cmp_melt, x='Model', y='Score', hue='Metric', palette='muted')
    plt.title('Original Training vs Trainset Ablation')
    plt.xticks(rotation=30)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("Original benchmark.csv not found. Comparison skipped.")

print("\n=== ABLATION STUDY COMPLETE ===")